# Football Detection Training — Kaggle Version

This notebook trains **YOLO11m** on the Roboflow `soccer_detection v3` dataset.

Classes:
- ball
- goalkeeper
- player
- referee

## 0) Kaggle setup

Before running:
1. Open Kaggle Notebook settings.
2. Turn **Internet = On**.
3. Turn **Accelerator = GPU T4/P100** if available.
4. Add your Roboflow API key below.

In [1]:
!pip install -q ultralytics roboflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 74.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26

In [2]:
from ultralytics import YOLO
from roboflow import Roboflow
import os, yaml, shutil

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 1) Download dataset from Roboflow

In [3]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="SY5CeA6j2YkcGsxbJ2iF")
project = rf.workspace("shootaai").project("soccer_detection")
version = project.version(3)
dataset = version.download("yolov11")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to soccer_detection-3 in yolov11:: 100%|██████████| 4290/4290 [00:00<00:00, 7016.27it/s]


## 2) Verify `data.yaml`

In [4]:
data_yaml = os.path.join(dataset.location, "data.yaml")

with open(data_yaml, "r") as f:
    data = yaml.safe_load(f)

print(data)
print("Names:", data["names"])
print("Number of classes:", data["nc"])

{'train': '../train/images', 'val': '../valid/images', 'test': '../test/images', 'nc': 4, 'names': ['ball', 'goalkeeper', 'player', 'referee'], 'roboflow': {'workspace': 'shootaai', 'project': 'soccer_detection', 'version': 3, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/shootaai/soccer_detection/dataset/3'}}
Names: ['ball', 'goalkeeper', 'player', 'referee']
Number of classes: 4


## 3) Train YOLO11m

In [5]:
model = YOLO("yolo11m.pt")

results = model.train(
    data=data_yaml,
    epochs=100,
    imgsz=1280,
    batch=-1,
    patience=25,
    device=0,
    workers=2,
    cache=False,

    save=True,
    save_period=5,   # يحفظ checkpoint كل epoch
    exist_ok=True,   # لو نفس الاسم موجود ميعملش error

    project="/kaggle/working/runs",
    name="football_yolo11m_1280"
)

Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/soccer_detection-3/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=football_yolo11m_1280, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ove

## 4) Validate best model

In [6]:
best_model_path = "/kaggle/working/runs/football_yolo11m_1280/weights/best.pt"

model = YOLO(best_model_path)

metrics = model.val(
    data=data_yaml,
    imgsz=1280,
    device=0
)

Ultralytics 8.4.66 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 126 layers, 20,033,116 parameters, 0 gradients, 67.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1198.7±302.8 MB/s, size: 64.7 KB)
val: Scanning /kaggle/working/soccer_detection-3/valid/labels.cache... 333 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 333/333 82.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 21/21 2.2s/it 46.9s
                   all        333       7086      0.792      0.856      0.847      0.576
                  ball         97         97      0.646      0.676      0.619      0.292
            goalkeeper        170        177      0.738      0.893      0.864      0.611
                player        333       6178      0.931      0.977      0.986      0.755
               referee        315        634      0.852       0.88      0.917      0.647
Speed: 6.7ms preprocess, 

## 6) Zip outputs for download

In [7]:
shutil.make_archive(
    "/kaggle/working/football_yolo11m_outputs",
    "zip",
    "/kaggle/working/runs/football_yolo11m_1280"
)

print("Best model:", best_model_path)
print("Zip file: /kaggle/working/football_yolo11m_outputs.zip")

Best model: /kaggle/working/runs/football_yolo11m_1280/weights/best.pt
Zip file: /kaggle/working/football_yolo11m_outputs.zip


## 5) Test on images or video

In [8]:
# Upload a video/image to Kaggle input, then change this path.
SOURCE = "/kaggle/input/YOUR_DATASET_OR_VIDEO/test_video.mp4"

model.predict(
    source=SOURCE,
    imgsz=1280,
    conf=0.25,
    save=True,
    project="/kaggle/working/predictions",
    name="football_test"
)

FileNotFoundError: /kaggle/input/YOUR_DATASET_OR_VIDEO/test_video.mp4 does not exist

In [ ]:
import shutil, os

best_model_path = "/kaggle/working/runs/football_yolo11m_1280/weights/best.pt"

shutil.copy(best_model_path, "/kaggle/working/best.pt")

shutil.make_archive(
    "/kaggle/working/football_model_final",
    "zip",
    "/kaggle/working/runs/football_yolo11m_1280"
)

print("Ready to download:")
print("/kaggle/working/best.pt")
print("/kaggle/working/football_model_final.zip")

## If Kaggle GPU memory fails

Change training to:

```python
imgsz=960
batch=8
```

or:

```python
imgsz=960
batch=-1
```